# 🏥 AI Clinical Scribe — Audio Pipeline (Google Colab)

**Chạy Whisper STT + pyannote Diarization trên Colab GPU (T4 free)**

> **Trước khi chạy:** Vào `Runtime → Change runtime type → T4 GPU`

## Luồng
```
File audio (.wav)
    → Whisper STT (tiếng Việt)   [Cell 3]
    → pyannote Diarization        [Cell 4]
    → Merge → JSON turns          [Cell 5]
    → In kết quả SOAP (mock)      [Cell 6]
```

## Cell 1 — Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần thiết
# Lần đầu chạy mất ~3-5 phút
!pip install -q transformers accelerate
!pip install -q pyannote.audio
!pip install -q librosa soundfile
!pip install -q ffmpeg-python
!apt-get install -q ffmpeg

print('✅ Cài đặt xong!')

## Cell 2 — Upload file audio + nhập HuggingFace token

In [ ]:
from google.colab import files
import os

# ── Upload file audio ──────────────────────────────────────
print('Chọn file audio (.wav hoặc .mp3) để upload:')
uploaded = files.upload()

AUDIO_FILE = list(uploaded.keys())[0]
print(f'\n✅ Đã upload: {AUDIO_FILE} ({os.path.getsize(AUDIO_FILE)/1024:.1f} KB)')

# ── HuggingFace token (cần cho pyannote) ──────────────────
# Lấy token tại: https://huggingface.co/settings/tokens
# Phải accept điều khoản tại: https://huggingface.co/pyannote/speaker-diarization-3.1
HF_TOKEN = 'hf_xxxxxxxxxxxxxxxxxxxx'  # <-- Thay token của bạn vào đây

if HF_TOKEN == 'hf_xxxxxxxxxxxxxxxxxxxx':
    print('\n⚠️  Chưa điền HF_TOKEN! Diarization sẽ bị bỏ qua.')
    print('   Lấy token tại: https://huggingface.co/settings/tokens')
else:
    print(f'\n✅ HF Token: {HF_TOKEN[:10]}...')

## Cell 3 — Bước 1: Whisper STT

In [ ]:
import torch
from transformers import pipeline as hf_pipeline

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU ✅" if device == 0 else "CPU (chậm hơn)"}')

# Load Whisper medium (~1.5GB, cache lại sau lần đầu)
print('\nĐang load Whisper medium...')
whisper_pipe = hf_pipeline(
    'automatic-speech-recognition',
    model='openai/whisper-medium',
    generate_kwargs={'language': 'vi', 'task': 'transcribe'},
    return_timestamps=True,
    device=device,
    chunk_length_s=30,
)
print('✅ Đã load Whisper!')

# Chạy STT
print(f'\nĐang transcribe {AUDIO_FILE}...')
stt_result = whisper_pipe(AUDIO_FILE)

print(f'\n✅ STT xong! {len(stt_result["text"])} ký tự')
print(f'\nTranscript:\n{stt_result["text"]}')
print(f'\nSố chunks: {len(stt_result.get("chunks", []))}')

## Cell 4 — Bước 2: pyannote Diarization (ai nói đoạn nào)

In [ ]:
diarization = None

if HF_TOKEN == 'hf_xxxxxxxxxxxxxxxxxxxx':
    print('⚠️  Bỏ qua diarization (chưa có HF token)')
else:
    from pyannote.audio import Pipeline as PyannotePipeline

    print('Đang load pyannote speaker-diarization-3.1...')
    diarize_model = PyannotePipeline.from_pretrained(
        'pyannote/speaker-diarization-3.1',
        use_auth_token=HF_TOKEN
    )
    diarize_model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

    print(f'Đang diarize {AUDIO_FILE}...')
    diarization = diarize_model(AUDIO_FILE)

    print('\n✅ Diarization xong! Kết quả:')
    for segment, _, speaker in diarization.itertracks(yield_label=True):
        print(f'  [{segment.start:.1f}s → {segment.end:.1f}s] {speaker}')

## Cell 5 — Bước 3: Merge STT + Diarization → JSON turns

In [ ]:
import json

def merge_stt_and_diarization(chunks, diarization):
    """Ghép Whisper chunks với pyannote speaker labels."""
    turns = []
    for chunk in chunks:
        start, end = chunk['timestamp']
        if start is None or end is None:
            continue
        speaker_votes = {}
        for segment, _, speaker in diarization.itertracks(yield_label=True):
            overlap_start = max(start, segment.start)
            overlap_end   = min(end, segment.end)
            if overlap_end > overlap_start:
                duration = overlap_end - overlap_start
                speaker_votes[speaker] = speaker_votes.get(speaker, 0) + duration
        dominant = max(speaker_votes, key=speaker_votes.get) if speaker_votes else 'unknown'
        turns.append({'speaker': dominant, 'text': chunk['text'].strip(), 'start': start, 'end': end})
    return turns


# Nếu có diarization → merge; nếu không → dùng raw text
chunks = stt_result.get('chunks', [])

if diarization is not None and chunks:
    turns = merge_stt_and_diarization(chunks, diarization)
    print(f'✅ Merge xong! {len(turns)} turns')
else:
    # Fallback: không có diarization → 1 turn duy nhất
    turns = [{'speaker': 'unknown', 'text': stt_result['text'], 'start': 0, 'end': 0}]
    print('⚠️  Không có diarization → 1 turn, speaker = unknown')

print('\nDanh sách turns:')
for t in turns:
    print(f'  [{t["speaker"]}] {t["text"][:60]}...')

## Cell 6 — Map speaker labels + Xuất JSON

In [ ]:
from datetime import datetime

# Map speaker labels
unique_speakers = list(set(t['speaker'] for t in turns))
print(f'Phát hiện {len(unique_speakers)} người nói: {unique_speakers}')
print('\nHãy map từng speaker → vai trò:')

mapping = {}
role_options = ['doctor', 'patient', 'family', 'other']
for spk in unique_speakers:
    sample = next(t['text'] for t in turns if t['speaker'] == spk)
    print(f'\n{spk}: "{sample[:80]}"')
    role = input(f'Vai trò ({"/".join(role_options)}): ').strip().lower()
    mapping[spk] = role if role in role_options else 'other'

for turn in turns:
    turn['speaker'] = mapping.get(turn['speaker'], 'other')

# Xuất JSON
output = {
    'session_id': f'VNM-{datetime.now().strftime("%Y%m%d")}-COLAB',
    'patient_id': 'BN-2026-00001',
    'recorded_at': datetime.now().isoformat(),
    'source': 'whisper-medium + pyannote-3.1',
    'turns': turns
}

output_file = 'raw_transcript_colab.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'\n✅ Đã lưu: {output_file}')
print(json.dumps(output, ensure_ascii=False, indent=2))

## Cell 7 — Download kết quả về máy

In [ ]:
from google.colab import files

files.download('raw_transcript_colab.json')
print('✅ Download thành công! Dùng file này làm input cho medical_agent/main.py')

## Ghi chú

| Bước | Thời gian (GPU T4) |
|------|--------------------|
| Cài thư viện | ~3-5 phút (lần đầu) |
| Load Whisper medium | ~1-2 phút (download ~1.5GB) |
| STT audio 5 phút | ~30 giây |
| Diarization audio 5 phút | ~1 phút |

**Lấy HuggingFace token:**
1. Tạo tài khoản tại https://huggingface.co
2. Vào https://huggingface.co/pyannote/speaker-diarization-3.1 → Accept điều khoản
3. Tạo token tại https://huggingface.co/settings/tokens
4. Dán vào ô `HF_TOKEN` ở Cell 2